In [1]:
import re
import csv

# مسیرها
input_path = r"F:\Thesis\project\2-RAG\raw_laws\civil\omoor_hasabi.txt"
output_tsv = r"F:\Thesis\project\2-RAG\raw_laws\civil\omoor_hasabi_articles.tsv"

LAW_CODE = "omoor_hasabi"
LAW_NAME = "قانون امور حسبی"


def normalize_digits(text: str) -> str:
    persian_digits = "۰۱۲۳۴۵۶۷۸۹"
    english_digits = "0123456789"
    trans_table = str.maketrans(persian_digits, english_digits)
    return text.translate(trans_table)


def normalize_persian(text: str) -> str:
    text = text.replace("ي", "ی").replace("ك", "ک")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


# 1) خواندن و نرمال‌سازی اولیه
with open(input_path, "r", encoding="utf-8") as f:
    raw = f.read()

raw = normalize_digits(raw)
raw = raw.replace("–", "-").replace("—", "-")

# 2) شکستن خطوط: قبل از «باب»، «فصل»، «ماده» در وسط خط، newline بگذاریم
raw = re.sub(r"(?<!^)(باب\s+[^\n]+)", r"\n\1", raw, flags=re.MULTILINE)
raw = re.sub(r"(?<!^)(فصل\s+[^\n]+)", r"\n\1", raw, flags=re.MULTILINE)
raw = re.sub(r"(?<!^)(ماده\s+\d+)", r"\n\1", raw, flags=re.MULTILINE)

lines = raw.splitlines()

# 3) الگوها: باب، فصل، ماده
# مثال:
#   باب اول- در کلیات
#   فصل اول– صلاحیت دادگاه قیمومت
#   ماده 1– امور حسبی ...
book_pattern    = re.compile(r"^\s*باب\s+(.+)$")
chapter_pattern = re.compile(r"^\s*فصل\s+(.+)$")
article_pattern = re.compile(r"^\s*ماده\s+(\d+)\s*[-–]\s*(.*)$")

current_book = ""
current_chapter = ""
articles = []

current_article_num = None
current_article_text = ""


for line in lines:
    stripped = line.strip()
    if not stripped:
        continue

    # 1) تشخیص باب
    m_book = book_pattern.match(stripped)
    if m_book:
        # نهایی کردن ماده باز (در صورت وجود)
        if current_article_num is not None and current_article_text:
            one_line = normalize_persian(current_article_text)
            articles.append(
                {
                    "law_code": LAW_CODE,
                    "law_name": LAW_NAME,
                    "book": current_book,
                    "chapter": current_chapter,
                    "article_number": current_article_num,
                    "text": one_line,
                }
            )
            current_article_num = None
            current_article_text = ""

        current_book = normalize_persian(stripped)
        # وقتی باب عوض می‌شود، فصل را ریست می‌کنیم (چون فصل‌های جدید مخصوص باب جدید هستند)
        current_chapter = ""
        continue

    # 2) تشخیص فصل
    m_chap = chapter_pattern.match(stripped)
    if m_chap:
        if current_article_num is not None and current_article_text:
            one_line = normalize_persian(current_article_text)
            articles.append(
                {
                    "law_code": LAW_CODE,
                    "law_name": LAW_NAME,
                    "book": current_book,
                    "chapter": current_chapter,
                    "article_number": current_article_num,
                    "text": one_line,
                }
            )
            current_article_num = None
            current_article_text = ""

        current_chapter = normalize_persian(stripped)
        continue

    # 3) تشخیص شروع ماده
    m_art = article_pattern.match(stripped)
    if m_art:
        if current_article_num is not None and current_article_text:
            one_line = normalize_persian(current_article_text)
            articles.append(
                {
                    "law_code": LAW_CODE,
                    "law_name": LAW_NAME,
                    "book": current_book,
                    "chapter": current_chapter,
                    "article_number": current_article_num,
                    "text": one_line,
                }
            )

        num_str = m_art.group(1)
        rest_text = m_art.group(2).strip()

        try:
            current_article_num = int(num_str)
        except ValueError:
            current_article_num = None

        if rest_text:
            current_article_text = f"ماده {num_str}- {rest_text}"
        else:
            current_article_text = f"ماده {num_str}-"
        continue

    # 4) ادامهٔ متن مادهٔ جاری
    if current_article_num is not None:
        current_article_text += " " + stripped

# 5) بستن آخرین ماده
if current_article_num is not None and current_article_text:
    one_line = normalize_persian(current_article_text)
    articles.append(
        {
            "law_code": LAW_CODE,
            "law_name": LAW_NAME,
            "book": current_book,
            "chapter": current_chapter,
            "article_number": current_article_num,
            "text": one_line,
        }
    )

# 6) نوشتن خروجی TSV
fieldnames = ["law_code", "law_name", "book", "chapter", "article_number", "text"]

with open(output_tsv, "w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames, delimiter="\t")
    writer.writeheader()
    for row in articles:
        writer.writerow(row)

print("✓ پردازش قانون امور حسبی تمام شد.")
print(f"✓ تعداد مواد استخراج شده: {len(articles)}")
print(f"✓ خروجی TSV: {output_tsv}")


✓ پردازش قانون امور حسبی تمام شد.
✓ تعداد مواد استخراج شده: 378
✓ خروجی TSV: F:\Thesis\project\2-RAG\raw_laws\civil\omoor_hasabi_articles.tsv
